# Data Mining Techniques Assignment 1

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [12]:
df = pd.read_csv('dataset_mood_smartphone.csv')


## 1. Data Quality & Logical Structure

In [ ]:
print(df.shape)          
print(df.info())         
print(df.head())

(376912, 5)
<class 'pandas.DataFrame'>
RangeIndex: 376912 entries, 0 to 376911
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Unnamed: 0  376912 non-null  int64  
 1   id          376912 non-null  str    
 2   time        376912 non-null  str    
 3   variable    376912 non-null  str    
 4   value       376710 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 14.4 MB
None
   Unnamed: 0       id                     time variable  value
0           1  AS14.01  2014-02-26 13:00:00.000     mood    6.0
1           2  AS14.01  2014-02-26 15:00:00.000     mood    6.0
2           3  AS14.01  2014-02-26 18:00:00.000     mood    6.0
3           4  AS14.01  2014-02-26 21:00:00.000     mood    7.0
4           5  AS14.01  2014-02-27 09:00:00.000     mood    6.0


From info():
- 376912 records
- 4 attributes: row_number, id, time, variable, value.
(1) id = str
(2) time = str
(3) variable = str
(4) value = float64
(5) Unnamed = int64
- 5 columns
- 202 values missing in the value category

From this, we can understand that the dataset is a multivariat time series, where more than one variable is tracked over time. Each row is one measurement event with:
- id = participant
- time = timestamp
- variable = which signal was measured
- value = observed value
- unnamed = idex column

## 2. Data Exploration

In [25]:
n_rows = len(df)
n_cols = df.shape[1]
n_participants = df["id"].nunique()
n_variables = df["variable"].nunique()
time_min = df["time"].min()
time_max = df["time"].max()
missing_value_total = df["value"].isna().sum()

print("Rows:", n_rows)
print("Columns:", n_cols)
print("Unique participants:", n_participants)
print("Unique variables:", n_variables)
print("Time range:", time_min, "to", time_max)
print("Missing values in value:", missing_value_total)

Rows: 376912
Columns: 5
Unique participants: 27
Unique variables: 19
Time range: 2014-02-17 07:00:52.197 to 2014-06-09 00:00:00.000
Missing values in value: 202


In [26]:
variables = sorted(df["variable"].unique())
print("Variables:")
for v in variables:
    print(v)

Variables:
activity
appCat.builtin
appCat.communication
appCat.entertainment
appCat.finance
appCat.game
appCat.office
appCat.other
appCat.social
appCat.travel
appCat.unknown
appCat.utilities
appCat.weather
call
circumplex.arousal
circumplex.valence
mood
screen
sms


In [ ]:
counts_per_variable = df["variable"].value_counts()
plt.figure(figsize=(12, 5))
counts_per_variable.plot(kind="bar")
plt.title("Number of observations per variable")
plt.xlabel("Variable")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

variable
screen                  96578
appCat.builtin          91288
appCat.communication    74276
appCat.entertainment    27125
activity                22965
appCat.social           19145
appCat.other             7650
circumplex.arousal       5643
circumplex.valence       5643
appCat.office            5642
mood                     5641
call                     5239
appCat.travel            2846
appCat.utilities         2487
sms                      1798
appCat.finance            939
appCat.unknown            939
appCat.game               813
appCat.weather            255
Name: count, dtype: int64

In [28]:
counts_per_participant = df["id"].value_counts()
print(counts_per_participant)

id
AS14.01    21999
AS14.23    21852
AS14.13    19592
AS14.28    19276
AS14.06    18092
AS14.29    17499
AS14.12    17311
AS14.30    17279
AS14.26    16403
AS14.33    16390
AS14.07    16045
AS14.17    15826
AS14.05    15745
AS14.02    14581
AS14.27    14575
AS14.24    14430
AS14.03    14425
AS14.25    12589
AS14.31    11889
AS14.19    11397
AS14.32    11193
AS14.09    10886
AS14.14     9286
AS14.08     7902
AS14.16     3982
AS14.20     3620
AS14.15     2848
Name: count, dtype: int64


In [29]:
variable_stats = (
    df.groupby("variable")["value"]
      .agg(["count", "min", "median", "mean", "std", "max"])
      .sort_values("count", ascending=False)
)

print(variable_stats.round(3))

                      count        min  median     mean      std        max
variable                                                                   
screen                96578      0.035  20.044   75.335  253.822   9867.007
appCat.builtin        91288 -82798.871   4.038   18.538  415.989  33960.246
appCat.communication  74276      0.006  16.226   43.344  128.913   9830.777
appCat.entertainment  27125     -0.011   3.391   37.576  262.960  32148.677
activity              22965      0.000   0.022    0.116    0.187      1.000
appCat.social         19145      0.094  28.466   72.402  261.552  30000.906
appCat.other           7650      0.014  10.028   25.811  112.781   3892.038
appCat.office          5642      0.003   3.106   22.579  449.601  32708.818
mood                   5641      1.000   7.000    6.993    1.033     10.000
circumplex.arousal     5597     -2.000   0.000   -0.099    1.052      2.000
circumplex.valence     5487     -2.000   1.000    0.688    0.671      2.000
call        